# Credit Scoring — Fase 5: Scorecard Interpretable (WoE/IV)

Esta fase implementa el **estándar de la industria bancaria**: una scorecard que convierte el modelo en un sistema de puntos auditable y regulable.  

**Fórmula de escala:**
```
Score = A - B × ln(odds)
Puntos_variable_bin = -(WoE × β_i × B)
```
donde `A = 600` (score base), `B = PDO / ln(2)` con `PDO = 20` (cada 20 puntos doblan los odds).

**Bandas de riesgo:**
- Score < 550 → Alto riesgo → Rechazar
- 550–650 → Riesgo medio → Revisión manual
- Score > 650 → Bajo riesgo → Aprobar

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
sys.path.append('..')
from src.features.woe_encoder import WoEEncoder
from src.models.scorecard import Scorecard

DATA_PROC = Path('../data/processed')
MODELS    = Path('../models/saved')
FIGURES   = Path('../reports/figures')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Carga de datos y encoder WoE

In [ ]:
train_woe = pd.read_csv(DATA_PROC / 'train_woe.csv')
val_woe   = pd.read_csv(DATA_PROC / 'val_woe.csv')
test_woe  = pd.read_csv(DATA_PROC / 'test_woe.csv')
train_raw = pd.read_csv(DATA_PROC / 'train_raw.csv')
test_raw  = pd.read_csv(DATA_PROC / 'test_raw.csv')

X_train_woe, y_train = train_woe.drop('target', axis=1), train_woe['target']
X_val_woe,   y_val   = val_woe.drop('target', axis=1),   val_woe['target']
X_test_woe,  y_test  = test_woe.drop('target', axis=1),  test_woe['target']
X_test_raw           = test_raw.drop('target', axis=1)

woe_encoder: WoEEncoder = joblib.load(MODELS / 'woe_encoder.joblib')

print(f'Train WoE: {X_train_woe.shape}')
print(f'Features: {list(X_train_woe.columns)}')

## 2. Logistic Regression sobre WoE

La regresión logística sobre WoE es la base de la scorecard. La linealidad del WoE garantiza que los coeficientes sean interpretables como contribución de cada variable al score.

In [ ]:
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
logreg.fit(X_train_woe, y_train)

y_prob_val  = logreg.predict_proba(X_val_woe)[:, 1]
y_prob_test = logreg.predict_proba(X_test_woe)[:, 1]

auc_val  = roc_auc_score(y_val, y_prob_val)
auc_test = roc_auc_score(y_test, y_prob_test)

print(f'AUC-ROC validación: {auc_val:.4f}')
print(f'AUC-ROC test:       {auc_test:.4f}')

coef_df = pd.DataFrame({
    'feature': X_train_woe.columns,
    'beta':    logreg.coef_[0]
}).sort_values('beta', key=abs, ascending=False)
display(coef_df)

## 3. Construcción de la scorecard

Convertimos cada (variable, bin) en puntos enteros. Los puntos son **aditivos**: el score final de un cliente es la suma de los puntos de todos sus bins.

In [ ]:
scorecard = Scorecard(base_score=600, base_odds=1/50, pdo=20)
scorecard.fit(logreg, woe_encoder)

print(f'Score base (intercepto): {scorecard.intercept_points_:.1f}')
print(f'Factor B (PDO/ln2):      {scorecard.B:.4f}')
print(f'Factor A:                {scorecard.A:.4f}')
print(f'\nTotal de reglas en la scorecard: {len(scorecard.scorecard_table_)}')
display(scorecard.scorecard_table_.head(15))

## 4. Tabla scorecard exportable

Formato que puede entregarse a equipos de negocio, sistemas de decisión o reguladores.

In [ ]:
export_cols = ['variable', 'bin_label', 'woe', 'points_int']
scorecard_export = scorecard.scorecard_table_[export_cols].copy()
scorecard_export.columns = ['Variable', 'Bin', 'WoE', 'Puntos']

# Guardar
scorecard_export.to_csv(MODELS / 'scorecard_table.csv', index=False)

# Mostrar por variable
for var, group in scorecard_export.groupby('Variable', sort=False):
    print(f'\n── {var} ──')
    display(group.reset_index(drop=True))

## 5. Visualización: puntos por variable

Rango de puntos de cada variable (máx - mín). Variables con mayor rango tienen más influencia en el score final.

In [ ]:
pt = scorecard.scorecard_table_.groupby('variable')['points_int'].agg(['min', 'max'])
pt['range'] = pt['max'] - pt['min']
pt = pt.sort_values('range', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(pt))

for i, (var, row) in enumerate(pt.iterrows()):
    ax.barh(i, row['max'] - row['min'], left=row['min'],
            color='steelblue', alpha=0.7, edgecolor='white', height=0.6)
    ax.text(row['min'] - 1, i, f"{int(row['min'])}", va='center', ha='right', fontsize=8)
    ax.text(row['max'] + 1, i, f"{int(row['max'])}", va='center', ha='left', fontsize=8)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(pt.index, fontsize=9)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('Puntos')
ax.set_title('Rango de puntos por variable (scorecard)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / '14_scorecard_points_range.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap de puntos por variable × bin
pivot = scorecard.scorecard_table_.pivot_table(
    index='variable', columns='bin_index', values='points_int', aggfunc='first'
)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Puntos de scorecard por variable y bin', fontsize=12)
ax.set_xlabel('Bin index (menor → mayor valor)')
plt.tight_layout()
plt.savefig(FIGURES / '15_scorecard_heatmap.png', bbox_inches='tight')
plt.show()

## 6. Distribución de scores en el set de test

Verificamos que la distribución de scores sea razonable y que las bandas de riesgo capturen la separación entre buenos y malos pagadores.

In [ ]:
scores_df = scorecard.score_batch(X_test_raw)
scores_df['target'] = y_test.values

print('Distribución de scores (test):')
print(scores_df['score'].describe().round(1).to_string())

print('\nDistribución por banda de riesgo:')
band_summary = scores_df.groupby('risk_band').agg(
    n=('score', 'count'),
    default_rate=('target', 'mean'),
    score_mean=('score', 'mean'),
).round(3)
display(band_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de scores por clase
for target_val, color, label in [(0, 'steelblue', 'No Default'), (1, 'tomato', 'Default')]:
    subset = scores_df[scores_df['target'] == target_val]['score']
    axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=label, density=True)

for thresh, ls in [(550, '--'), (650, '--')]:
    axes[0].axvline(thresh, color='black', linestyle=ls, linewidth=1)

axes[0].set_xlabel('Score')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de scores por clase')
axes[0].legend()

# Tasa de default por decil de score
scores_df['score_decil'] = pd.qcut(scores_df['score'], q=10, duplicates='drop')
default_by_decil = scores_df.groupby('score_decil', observed=True)['target'].mean()
default_by_decil.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Decil de score (menor → mayor riesgo)')
axes[1].set_ylabel('Tasa de default')
axes[1].set_title('Tasa de default por decil de score')
axes[1].set_xticklabels([str(i+1) for i in range(len(default_by_decil))], rotation=0)
axes[1].axhline(y_test.mean(), color='tomato', linestyle='--', linewidth=1, label='Promedio global')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / '16_score_distribution.png', bbox_inches='tight')
plt.show()

## 7. Función `calcular_score` — demostración con perfiles reales

In [ ]:
# Perfiles de ejemplo extraídos del dataset
perfiles = {
    'cliente_bajo_riesgo': {
        'revolving_util': 0.05, 'age': 52, 'late_30_59': 0, 'debt_ratio': 0.15,
        'monthly_income': 8000, 'open_credit_lines': 8, 'late_90plus': 0,
        'real_estate_loans': 1, 'late_60_89': 0, 'n_dependents': 2,
        'debt_to_income': 1200, 'total_late_payments': 0, 'income_per_dependent': 2667,
    },
    'cliente_borderline': {
        'revolving_util': 0.55, 'age': 38, 'late_30_59': 1, 'debt_ratio': 0.45,
        'monthly_income': 4500, 'open_credit_lines': 6, 'late_90plus': 0,
        'real_estate_loans': 1, 'late_60_89': 0, 'n_dependents': 2,
        'debt_to_income': 2025, 'total_late_payments': 1, 'income_per_dependent': 1500,
    },
    'cliente_alto_riesgo': {
        'revolving_util': 0.92, 'age': 29, 'late_30_59': 3, 'debt_ratio': 0.80,
        'monthly_income': 2200, 'open_credit_lines': 4, 'late_90plus': 2,
        'real_estate_loans': 0, 'late_60_89': 1, 'n_dependents': 3,
        'debt_to_income': 1760, 'total_late_payments': 6, 'income_per_dependent': 550,
    },
}

for nombre, perfil in perfiles.items():
    resultado = scorecard.score(perfil)
    print(f'\n══ {nombre.upper()} ══')
    print(f"  Score:             {resultado['score']}")
    print(f"  Prob. default:     {resultado['probability_default']:.2%}")
    print(f"  Decisión:          {resultado['decision']}")
    print(f"  Banda de riesgo:   {resultado['risk_band']}")
    print('  Top 3 factores:')
    for f in resultado['breakdown'][:3]:
        print(f"    {f['feature']:25s} | valor={f['value']:8.2f} | puntos={f['points']:+d}")

## 8. Serialización del scorecard

In [ ]:
scorecard.save(MODELS / 'scorecard.joblib')
joblib.dump(logreg, MODELS / 'logreg_woe.joblib')

print('Archivos guardados:')
print('  models/saved/scorecard.joblib')
print('  models/saved/scorecard_table.csv')
print('  models/saved/logreg_woe.joblib')

## Resumen Fase 5

**Scorecard construida con:**
- Logistic Regression sobre WoE-encoded features
- Escala estándar bancaria: A=600, PDO=20, odds base=1:50
- Puntos enteros por bin, aditivos y auditables

**Bandas de decisión:** < 550 Rechazar | 550–650 Manual | > 650 Aprobar

**Outputs:**
- `models/saved/scorecard.joblib` — objeto Scorecard serializado
- `models/saved/scorecard_table.csv` — tabla exportable
- `reports/figures/14–16` — visualizaciones de la scorecard

**Próximo paso:** `06_model_explainability_shap.ipynb`